# Task 7 — Discretization Error and Model Mismatch

**Research project:** Numerical Solution and PDE-Constrained Optimization of the Two-Dimensional Heat Equation

We generate synthetic temperatures on a **72×72** grid and estimate 25 Gaussian-source coefficients on **18×18, 24×24, 36×36, and 48×48** grids. We compare in-grid observations, cross-grid observations, additive noise, and a source with an additional localized Gaussian bump that is not included in the reconstruction basis.

**Important:** A finite regularization parameter can produce reconstruction error even when the data are perfectly aligned and noise-free. Therefore we separately compute the forward-observation consistency error. All displayed numbers are specific to these simulated settings.

The notebook writes `phase7_discretization_model_mismatch.png`, `phase7_results.npz` and `phase7_results_summary.json` in the current directory.

In [1]:
import numpy as np, matplotlib.pyplot as plt, json, time
from scipy.ndimage import map_coordinates
from pathlib import Path
P=Path('.'); rng=np.random.default_rng(20261017)
alpha=.1; T=.1; times=np.array([.025,.05,.1]); centers=np.array([(x,y) for y in np.linspace(.14,.86,5) for x in np.linspace(.14,.86,5)]); width=.105
sx=np.linspace(.12,.88,4); sy=np.linspace(.12,.88,4); sensors=np.array([(x,y) for y in sy for x in sx]); m=48

def fields(n):
 x=np.linspace(0,1,n+1); X,Y=np.meshgrid(x,x,indexing='ij')
 phi=np.stack([np.exp(-((X-a)**2+(Y-b)**2)/(2*width**2)) for a,b in centers],axis=0)
 return X,Y,phi

def solve(n,source):
 # source: [channels, nx+1, nx+1]
 dtmax=1/(4*alpha*n*n)
 q=int(np.ceil((T/.98/dtmax)/4))*4
 dt=T/q; ix=np.rint(times/dt).astype(int)
 assert np.allclose(ix*dt,times)
 u=np.zeros_like(source); arr=[]
 rx=alpha*dt*n*n
 for step in range(1,q+1):
  u[:,1:-1,1:-1]+=rx*(u[:,2:,1:-1]+u[:,:-2,1:-1]+u[:,1:-1,2:]+u[:,1:-1,:-2]-4*u[:,1:-1,1:-1])+dt*source[:,1:-1,1:-1]
  if step in ix: arr.append(u.copy())
 return np.stack(arr),dt,q

def observe(snaps,n):
 # [time,channels,x,y] -> sensor*time x channel, interpolation by coordinates
 cols=[]
 for t in range(len(times)):
  v=snaps[t]; ix=sensors[:,0]*n; iy=sensors[:,1]*n
  a=np.floor(ix).astype(int); b=np.floor(iy).astype(int); fx=ix-a;fy=iy-b
  vals=(v[:,a,b]*(1-fx)*(1-fy)+v[:,a+1,b]*fx*(1-fy)+v[:,a,b+1]*(1-fx)*fy+v[:,a+1,b+1]*fx*fy)
  cols.append(vals.T)
 return np.concatenate(cols,axis=0)

def reconstruct(A,d,lam):
 return np.linalg.solve(A.T@A+m*lam*np.eye(A.shape[1]), A.T@d)
def ferr(est,truth): return np.linalg.norm(est-truth)/np.linalg.norm(truth)

# true source with distributed coefficient pattern
c0=np.array([1.0,.5,.8,1.05,.3,.55,.9,1.3,.6,.7,.45,1.2,.7,.8,1.1,.65,.3,1.1,.95,.55,.9,.5,.75,1.25,.65])
# model mismatch: source outside span, localized asymmetric feature

def bump(X,Y):return 1.3*np.exp(-((X-.47)**2/(2*.035**2)+(Y-.37)**2/(2*.07**2)))

Xf,Yf,Pf=fields(72); Ftrue=np.einsum('i,ijk->jk',c0,Pf)
true_clean,dtf,nf=solve(72,np.stack([Ftrue,Ftrue+bump(Xf,Yf)]))
Dclean=observe(true_clean,72)
noise_std=.01*np.max(abs(Dclean[:,0])); noise=rng.normal(0,noise_std,size=m)
rows=[]; figures={}; forward_errors=[]
for n in [18,24,36,48]:
 X,Y,Phi=fields(n)
 responses,dt,nt=solve(n,Phi)
 A=observe(responses,n)
 forward_errors.append((n, float(ferr(A@c0,Dclean[:,0]))))
 _,s,_=np.linalg.svd(A,full_matrices=False)
 # balanced smoothing regularizer: fixed in units of data and s_max
 lam=(s[0]**2/m)*.002
 Fmatch=np.einsum('i,ijk->jk',c0,Phi)
 Fmis=Fmatch+bump(X,Y)
 for scenario in ['in_grid_clean','cross_grid_clean','cross_grid_noisy','mismatch_clean','mismatch_noisy']:
  if scenario.startswith('in_grid'): d=A@c0
  elif scenario.startswith('cross_grid'):d=Dclean[:,0].copy()
  else:d=Dclean[:,1].copy()
  if scenario.endswith('noisy'): d=d+noise
  coef=reconstruct(A,d,lam)
  field=np.einsum('i,ijk->jk',coef,Phi)
  target= Fmis if scenario.startswith('mismatch') else Fmatch
  rows.append(dict(n=n,scenario=scenario,source_error=ferr(field,target),coef_error=ferr(coef,c0),relative_residual=ferr(A@coef,d),cond=s[0]/s[-1],lam=lam,dt=dt,steps=nt))
  if n==36:figures[scenario]=(field,target,coef)
 print(n,nt, 'cond',round(s[0]/s[-1],1),[(r['scenario'],round(r['source_error']*100,2)) for r in rows if r['n']==n])

print('Forward observation errors:', forward_errors)
# separate regularization sweep for 36 model mismatch and aligned/noisy
n=36; X,Y,Phi=fields(n); snapshots,_,_=solve(n,Phi); A=observe(snapshots,n)
lam0=np.linalg.svd(A,compute_uv=False)[0]**2/m
sweep=[]
for scale in [0,1e-5,1e-4,1e-3,2e-3,1e-2,1e-1]:
 for case,d,target in [('matched',Dclean[:,0]+noise,np.einsum('i,ijk->jk',c0,Phi)),('mismatch',Dclean[:,1]+noise,np.einsum('i,ijk->jk',c0,Phi)+bump(X,Y))]:
  theta=np.linalg.lstsq(A,d,rcond=None)[0] if scale==0 else reconstruct(A,d,lam0*scale)
  field=np.einsum('i,ijk->jk',theta,Phi)
  sweep.append(dict(scale=scale,case=case,error=ferr(field,target)))

fig,axes=plt.subplots(2,3,figsize=(14,8),constrained_layout=True)
labels=['Reference source (not in basis)','Reconstructed source','Absolute difference']
tr=figures['mismatch_noisy'][1];pr=figures['mismatch_noisy'][0]
for j,(z,title) in enumerate(zip([tr,pr,np.abs(pr-tr)],labels)):
 im=axes[0,j].imshow(z.T,origin='lower',extent=[0,1,0,1],cmap='viridis');axes[0,j].set_title(title);fig.colorbar(im,ax=axes[0,j],shrink=.7)
for case,lbl in [('cross_grid_clean','Representable source: cross-grid, clean'),('cross_grid_noisy','Representable source: cross-grid, noisy'),('mismatch_noisy','Nonrepresentable source: noisy')]:
 a=[r for r in rows if r['scenario']==case];axes[1,0].plot([v['n'] for v in a],[100*v['source_error'] for v in a],marker='o',label=lbl)
axes[1,0].set(xlabel='Grid subdivisions per dimension',ylabel='Relative source error (%)',title='Inverse reconstruction vs grid');axes[1,0].legend(fontsize=7)
for case in ['matched','mismatch']:
 q=[r for r in sweep if r['case']==case];axes[1,1].semilogx([r['scale'] for r in q if r['scale']>0],[100*r['error'] for r in q if r['scale']>0],marker='o',label=case)
axes[1,1].set(xlabel='Regularization scale',ylabel='Relative source error (%)',title='Regularization and model mismatch');axes[1,1].legend()
for scenario in ['in_grid_clean','cross_grid_clean','mismatch_clean']:
 a=[r for r in rows if r['scenario']==scenario];axes[1,2].plot([r['n'] for r in a],[100*r['relative_residual'] for r in a],marker='o',label=scenario.replace('_',' '))
axes[1,2].set(xlabel='Grid subdivisions',ylabel='Relative observation residual (%)',title='Residuals do not measure source error');axes[1,2].legend(fontsize=8)
fig.suptitle('Task 7: Discretization error and source-model mismatch',fontweight='bold')
fig.savefig(P/'phase7_discretization_model_mismatch.png',dpi=170);plt.close(fig)
np.savez_compressed(P/'phase7_results.npz',rows_json=json.dumps(rows),sweep_json=json.dumps(sweep),forward_errors=np.array(forward_errors),true_coefficients=c0,sensor_xy=sensors,times=times,fine_reference_grid=72,noise_std=noise_std,reference_source=tr,reconstruction=pr,obs_clean=Dclean)
(P/'phase7_results_summary.json').write_text(json.dumps(dict(rows=rows,sweep=sweep,noise_std=noise_std),indent=2))


18 16 cond 2017.0 [('in_grid_clean', np.float64(4.06)), ('cross_grid_clean', np.float64(4.14)), ('cross_grid_noisy', np.float64(4.17)), ('mismatch_clean', np.float64(8.09)), ('mismatch_noisy', np.float64(7.92))]
24 24 cond 1987.0 [('in_grid_clean', np.float64(4.13)), ('cross_grid_clean', np.float64(4.23)), ('cross_grid_noisy', np.float64(4.19)), ('mismatch_clean', np.float64(8.29)), ('mismatch_noisy', np.float64(8.08))]
36 56 cond 2255.4 [('in_grid_clean', np.float64(4.13)), ('cross_grid_clean', np.float64(4.15)), ('cross_grid_noisy', np.float64(4.16)), ('mismatch_clean', np.float64(8.26)), ('mismatch_noisy', np.float64(8.07))]
48 96 cond 2267.3 [('in_grid_clean', np.float64(4.14)), ('cross_grid_clean', np.float64(4.15)), ('cross_grid_noisy', np.float64(4.16)), ('mismatch_clean', np.float64(8.26)), ('mismatch_noisy', np.float64(8.07))]
Forward observation errors: [(18, 0.001877224864385516), (24, 0.0019966169580751775), (36, 0.001084826739743316), (48, 0.0002318862574951996)]


7720